# The coverage of Figure 4's errors over 600 realizations (Sect. 5.2)

**In the paper:** Sect. 5.2, "Over 600 noise realizations with random sampling phases, the injected values fall within
the quoted 1, 2, and 3σ regions at the expected rates, and the quoted uncertainties of B's four parameters and of
both flux ratios match the scatter of the fitted values to within 4 %."

Fig. 4's measurement (three moving sources, the primary free, the prior on C's position) repeated on 600
realizations, each with its own sampling phases, centring error and catalogue error on C. Per realization: B's four
parameters relative to the primary, $\theta_{\rm rel} = \mathbf L\theta$ with covariance $\mathbf L\mathbf C\mathbf L^{\mathsf T}$, and the flux ratios
$\mathcal S_k = f_k / f_A$ with covariance $\mathbf J\mathbf C_f\mathbf J^{\mathsf T}$,
$\mathbf J = \partial(\mathcal S_B, \mathcal S_C)/\partial(f_A, f_B, f_C)$. The pull and coverage are defined as in
`sim_coverage_fig3.ipynb`.

| step | task | calls | paper |
|---|---|---|---|
| 1 | Fig. 4's scene, noise and scan law | `noise.gaia_like`, `sim.Scene`, `scanlaw.load_scan_law` | Sect. 5.2; Fig. 4 |
| 2 | one realization: the fit, the relative block and the flux ratios | `sim.random_zero_points`, `sim.simulate_campaign`, `joint.fit_joint` | Sect. 5.2 |
| 3 | the ensemble, stored; the first realizations recomputed | — | — |
| 4 | pulls and coverage | — | Sect. 5.2 |
| 5 | the pull distributions | — | — |

Every scanlib call carries the name of its module (`dyn.wls`, `joint.fit_joint`); the first code cell lists the
modules.

**What it reproduces** (Sect. 5.2). B's pull widths 0.97–0.99, the flux ratios' 0.96–0.99, all within 4 % of one;
coverage 0.693, 0.973, 1.000 (position alone 0.683, 0.967, 1.000). **Run time.** About 5 s; `FULL = True`
recomputes the 600 fits (14 min).

In [ ]:
# the imports: numpy, matplotlib, scipy's chi^2, and the scanlib modules; every scanlib call below carries its
# module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import noise                  # the noise model of a window
from scanlib import scanlaw                # scan angles, epochs and parallax factors
from scanlib import simulation as sim      # synthetic scenes and their windows
from scanlib import joint                  # the joint fit of all windows, and its helpers

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in sim_margins_lambda_w)
FWHM_MAS, PITCH_MAS, N_SAMPLES = 100.0, 59.0, 18
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)

# the switch FULL and the store of the ensemble
FULL = False                               # True: recompute the 600 fits (about 14 min)
STORE = Path("..") / "stored" / "fig4_coverage.npz"

## 1. Fig. 4's scene, noise and law (Sect. 5.2)

The per-sample noise of a G = 16 primary, the three sources (the primary; B at 75 mas, moving at 8 mas/yr
perpendicular to its separation; C at 250 mas, at rest), and the catalogue error assumed on C's position.

In [ ]:
# Figure 4's noise model: the per-sample noise of a G = 16 primary whose transit sums nine CCD crossings
PRIMARY_G = 16.0
BACKGROUND_E_PER_SAMPLE_PER_CCD = 10.0
# noise.gaia_like: in electrons sigma^2 = signal + 9 (b + r^2); returned per sample in the toy's flux units
NOISE_MODEL = noise.gaia_like(PRIMARY_G, sim.PRIMARY_FLUX, background=BACKGROUND_E_PER_SAMPLE_PER_CCD)

In [ ]:
# the three sources: the primary, B and C in polar form, B's speed, and the catalogue error assumed on C
# B, relative to the primary
RHO_B = 75.0                               # its separation [mas]
PA_B = 110.0                               # its position angle [deg], from north through east
Q_B = 0.2                                  # its flux ratio, B over the primary
# C, relative to the primary
RHO_C = 250.0                              # its separation [mas]
PA_C = 300.0                               # its position angle [deg]
Q_C = 0.3                                  # its flux ratio, C over the primary
PM_B_MAS_YR = 8.0                          # B's speed relative to the primary [mas/yr]
CAT_SIGMA_MAS = 5.0                        # ASSUMED catalogue error on C


def polar(rho_mas, pa_deg):
    """The sky offset of a point given by its separation and its position angle.

    Inputs
        rho_mas   the separation from the primary [mas]
        pa_deg    the position angle [deg], from north through east
    Returns an array of two numbers
        dra*      the offset to the east [mas]
        ddec      the offset to the north [mas]"""
    angle_rad = np.deg2rad(pa_deg)
    return np.array([rho_mas * np.sin(angle_rad), rho_mas * np.cos(angle_rad)])

In [ ]:
# the scene: the positions, B's motion, the fluxes; and the truth of B's four parameters
POS_B = polar(RHO_B, PA_B)                 # B's offset from the primary (dra*, ddec) [mas]
POS_C = polar(RHO_C, PA_C)                 # C's offset from the primary (dra*, ddec) [mas]
# B's motion: 8 mas/yr perpendicular to its separation
angle_B = np.deg2rad(PA_B)                 # B's position angle [rad]
PM_B = PM_B_MAS_YR * np.array([np.cos(angle_B), -np.sin(angle_B)])   # (mu_a*, mu_d) [mas/yr]
# sim.Scene: one row per source (the primary, B, C): the positions [mas], the fluxes, the motions [mas/yr]
positions = np.array([[0.0, 0.0], POS_B, POS_C])
fluxes = np.array([sim.PRIMARY_FLUX, Q_B * sim.PRIMARY_FLUX, Q_C * sim.PRIMARY_FLUX])
motions = np.array([[0.0, 0.0], PM_B, [0.0, 0.0]])        # the primary and C are at rest
SCENE = sim.Scene(positions, fluxes, pm=motions)
# np.r_ joins B's position and B's proper motion into one vector of four, the truth of B relative to the primary
TRUTH_B = np.r_[POS_B, PM_B]

In [ ]:
# the scan law, the windows' centring error, the two streams of realizations, and the levels of the coverage test
# psi: the scan angle of each transit [rad]; t: its epoch [yr from the reference epoch]
psi, t = scanlaw.load_scan_law(scanlaw.PLEIADES)
# the rms of the offset [mas] and of the residual motion [mas/yr], per component
CENTRING_MAS, CENTRING_MAS_YR = 0.3, 0.1   # the windows' centring error: offset, residual motion
SEEDS = (606, 707)                         # the seed of each stream of realizations
N_PER_SEED = 300                           # two streams of 300 realizations
LEVELS = (0.683, 0.954, 0.997)             # the probability inside the 1, 2, 3 sigma regions

## 2. One realization

The fit's parameters are four per source (primary, B, C); the prior holds
C's position at its catalogue value. B relative to the primary is the linear
map block_B − block_primary of θ and its covariance; the flux ratios and
their covariance follow from the fluxes' block by linear propagation.

In [ ]:
# Figure 4's measurement on one realization (one_draw)
def one_draw(rng):
    """Figure 4's measurement on one realization: the windows of the three sources, drawn with their noise, and
    the joint fit of the three moving sources with the primary free and the prior on C's position.

    Inputs
        rng   the random generator of the stream. One realization takes its draws from it in this order: the
              centring offset (2 numbers), the centring rate (2), the sampling phases (one per window), the noise
              of the windows, the catalogue error on C (2). The next realization goes on in the same stream
    Returns a table (a dictionary) with four entries
        "d"          B's four fitted parameters relative to the primary, minus the truth: (dra*, ddec) [mas] and
                     (mu_a*, mu_d) [mas/yr]; positive: the fit is above the truth
        "cov"        the quoted covariance of those four parameters, 4 x 4
        "ratio"      the two fitted flux ratios, (B over the primary, C over the primary)
        "ratio_cov"  the quoted covariance of the two flux ratios, 2 x 2"""
    # step 1, the centring error of the windows: the same offset [mas] added to every source's position and the
    # same rate [mas/yr] added to every source's motion
    centring_offset = rng.normal(0, CENTRING_MAS, 2)
    centring_rate = rng.normal(0, CENTRING_MAS_YR, 2)
    scene = sim.Scene(SCENE.positions + centring_offset, SCENE.fluxes, pm=SCENE.pm + centring_rate)

    # step 2, the data.
    # sim.random_zero_points: one sampling phase per window; sim.simulate_campaign: the noisy windows and each
    # sample's sigma (its third output, the sources' true along-scan positions, is omitted: the name _)
    eta0 = sim.random_zero_points(rng, psi.size, inst)
    windows, sigma, _ = sim.simulate_campaign(scene, (psi, t), rng, inst, NOISE_MODEL, eta0=eta0)

    # step 3, the catalogue position of C: its true offset from the primary plus the catalogue error
    catalogue_C = POS_C + rng.normal(0, CAT_SIGMA_MAS, 2)

    # step 4, the start of the fit.
    # the seed: the truth, every source at rest but B. np.r_ joins the three sources' blocks of four
    # (dra*, ddec, mu_a, mu_d): the primary, B, C
    seed = np.r_[np.zeros(4), POS_B, PM_B, POS_C, 0.0, 0.0]
    # the prior holds theta[8:10], C's position, at its catalogue value: (the indices in theta, the values, their
    # sigmas)
    prior = (np.array([8, 9]), catalogue_C, np.array([CAT_SIGMA_MAS] * 2))

    # step 5, the fit.
    # joint.fit_joint: the joint fit of all windows, three moving sources, the primary free, with the prior on C
    fit = joint.fit_joint(windows, psi, seed, inst, sigma, tau=t, prior=prior, free_primary=True, eta0=eta0)

    # step 6, B relative to the primary: theta_rel = L theta, with the matrix L (4 x 12) = B's block minus the
    # primary's
    lin = np.zeros((4, seed.size))                                            # L: B's block minus the primary's
    # the columns 4:8 are B's four parameters, the columns 0:4 the primary's; C's columns stay zero
    lin[:, 4:8] = np.eye(4)
    lin[:, 0:4] = -np.eye(4)
    theta_rel = lin @ fit["theta"]
    # the fit's covariance holds the 12 parameters, then the 3 fluxes: [:seed.size, :seed.size] keeps the
    # parameters' block, [-3:, -3:] the fluxes' block
    cov_rel = lin @ fit["cov"][:seed.size, :seed.size] @ lin.T

    # step 7, the flux ratios S_B = f_B / f_A and S_C = f_C / f_A (A: the primary), and their covariance J C_f J^T
    flux = np.asarray(fit["flux"], float)                  # (f_A, f_B, f_C)
    flux_cov = np.asarray(fit["cov"])[-3:, -3:]            # C_f: the covariance of the three fluxes
    # J = d(S_B, S_C) / d(f_A, f_B, f_C): row 1 is (-f_B, f_A, 0) / f_A^2, row 2 is (-f_C, 0, f_A) / f_A^2
    jac = np.array([[-flux[1], flux[0], 0.0], [-flux[2], 0.0, flux[0]]]) / flux[0] ** 2
    ratio = flux[1:] / flux[0]                             # the fluxes of B and C, each over the primary's
    return dict(d=theta_rel - TRUTH_B, cov=cov_rel, ratio=ratio, ratio_cov=jac @ flux_cov @ jac.T)

In [ ]:
# the measurement on the first n realizations of one stream (run)
def run(seed, n):
    """The measurement on the first n realizations of one stream.

    Inputs
        seed   the seed of the stream (606 or 707)
        n      the number of realizations
    Returns a table (a dictionary) with the four entries of one_draw(...); each entry is an array whose first index
    is the realization."""
    # one generator for the whole stream, seeded by the stream's seed alone: realization k takes its draws after
    # those of the realizations 0 to k - 1, so it cannot be recomputed without them
    rng = np.random.default_rng(seed)
    records = []
    for _ in range(n):                     # one pass per realization; the counter is not used
        records.append(one_draw(rng))
    stacked = {}
    for key in ("d", "cov", "ratio", "ratio_cov"):
        stacked[key] = np.array([record[key] for record in records])   # the entry of every realization, in order
    return stacked

## 3. The ensemble, and the check that the package still makes it

Two streams of 300 realizations, loaded from the store (`FULL = True` recomputes them); the first two of each
stream are recomputed and must equal the store bit for bit.

In [ ]:
# the ensemble of 2 x 300 realizations: recomputed and stored with FULL, otherwise loaded from the store
# ensemble: a table from an entry's name (KEYS, the four entries of one_draw) to its array over (stream,
# realization, ...)
KEYS = ("d", "cov", "ratio", "ratio_cov")
if FULL:
    # step 1, each stream on its own
    runs = []
    for stream_seed in SEEDS:
        runs.append(run(stream_seed, N_PER_SEED))
    # step 2, the streams stacked: np.stack puts the stream as the first index
    ensemble = {}
    for key in KEYS:
        ensemble[key] = np.stack([stream[key] for stream in runs])
    STORE.parent.mkdir(exist_ok=True)
    np.savez(STORE, **ensemble, seeds=np.array(SEEDS))
    print(f"computed and stored {STORE}")
else:
    with np.load(STORE) as stored:
        ensemble = {}
        for key in KEYS:
            ensemble[key] = stored[key]
        assert tuple(stored["seeds"]) == SEEDS             # the store was made with this notebook's seeds
    print(f"loaded {STORE}")
assert ensemble["d"].shape == (len(SEEDS), N_PER_SEED, 4)

In [ ]:
# the first 2 realizations of each stream recomputed: they must equal the store bit for bit
N_CHECK = 2
# the arrays are (stream, realization, ...): [i, :N_CHECK] keeps stream i and its first N_CHECK realizations
for i, stream_seed in enumerate(SEEDS):    # i: the stream's number
    fresh = run(stream_seed, N_CHECK)
    for key in KEYS:
        assert np.array_equal(fresh[key], ensemble[key][i, :N_CHECK]), \
            f"{key}: the package no longer reproduces the stored ensemble; set FULL = True"
print(f"first {N_CHECK} realizations of each stream recomputed: identical")

## 4. Pulls and coverage (Sect. 5.2)

B relative to the primary: the pull of each parameter, and the coverage of its 4-D covariance and of the
position alone; the two flux ratios: their pulls. The asserts are Sect. 5.2's statements.

In [ ]:
# truth: pulls and coverage of B relative to the primary (4-D and position alone), and pulls of the two flux
# ratios. reshape(-1, 4) joins the two streams: (2, 300, 4) becomes (600, 4); np.diagonal over the last two axes
# keeps each realization's variances
# step 1, B relative to the primary: fitted minus true, and the quoted covariance, one row per realization
offsets = ensemble["d"].reshape(-1, 4)
cov = ensemble["cov"].reshape(-1, 4, 4)
n = len(offsets)                           # the number of realizations
# step 2, the pulls: fitted minus true over the quoted error, the square root of the variance
quoted_error = np.sqrt(np.diagonal(cov, axis1=1, axis2=2))
pulls = offsets / quoted_error
# step 3, Delta chi2 = d^T C^-1 d of every realization, with d the offsets and C the covariance. The einsum sums
# d[n, i] C^-1[n, i, j] d[n, j] over i and j for each realization n
dchi2_4d = np.einsum("ni,nij,nj->n", offsets, np.linalg.inv(cov), offsets)
# the position alone: [:, :2] keeps the first two parameters of every realization, [:, :2, :2] their 2 x 2 block
position_offsets = offsets[:, :2]
position_cov = cov[:, :2, :2]
dchi2_position = np.einsum("ni,nij,nj->n", position_offsets, np.linalg.inv(position_cov), position_offsets)
# step 4, the coverage: the fraction of the realizations with Delta chi2 below the level of chi^2 (4 degrees of
# freedom; 2 for the position) that holds the probability 0.683, 0.954 or 0.997
coverage_4d = []
coverage_position = []
for probability in LEVELS:
    coverage_4d.append(np.mean(dchi2_4d < chi2.ppf(probability, 4)))
    coverage_position.append(np.mean(dchi2_position < chi2.ppf(probability, 2)))
coverage_4d = np.array(coverage_4d)
coverage_position = np.array(coverage_position)
# step 5, the pulls of the two flux ratios: fitted minus true (Q_B, Q_C), over the quoted error
ratio_offsets = ensemble["ratio"].reshape(-1, 2) - [Q_B, Q_C]
ratio_error = np.sqrt(np.diagonal(ensemble["ratio_cov"].reshape(-1, 2, 2), axis1=1, axis2=2))
ratio_pulls = ratio_offsets / ratio_error
# step 6, the pull widths: the standard deviation over the realizations (axis 0), per parameter
width_B = pulls.std(0)
width_S = ratio_pulls.std(0)
print(f"{n} realizations")
print(f"B relative to the primary: pull mean {np.round(pulls.mean(0), 2)}, width {np.round(width_B, 2)} "
      f"(+-{1 / np.sqrt(2 * n):.2f})")
print(f"    coverage 4-D {np.round(coverage_4d, 3)}, position alone {np.round(coverage_position, 3)}  "
      f"(expected {LEVELS})")
print(f"flux ratios S_B, S_C: pull mean {np.round(ratio_pulls.mean(0), 2)}, width {np.round(width_S, 2)}")

In [ ]:
# Sect. 5.2: the coverage at the expected rates (within 3 binomial sigma, plus 0.01);
# every quoted error matches the scatter to within 4 %
expected = np.array(LEVELS)
# the binomial sigma of a fraction p over n realizations is sqrt(p (1 - p) / n)
tolerance = 3 * np.sqrt(expected * (1 - expected) / n) + 0.01
assert np.all(np.abs(coverage_4d - expected) < tolerance) and np.all(np.abs(coverage_position - expected) < tolerance)
# np.r_ joins the four widths of B and the two of the flux ratios
assert np.all(np.abs(np.round(np.r_[width_B, width_S], 2) - 1.0) <= 0.04 + 1e-9)
# the coverage that the text quotes
assert np.array_equal(np.round(coverage_4d, 3), [0.693, 0.973, 1.0])
print("the quoted errors of Figure 4's measurement are honest")

## 5. The pull distributions

B's four parameters and the two flux ratios, against the standard normal.

In [ ]:
# the figure: the six pull distributions against the standard normal
names = [r"$\Delta\alpha^*_0$", r"$\Delta\delta_0$", r"$\mu_{\alpha^*}$", r"$\mu_\delta$",
         r"$\mathcal{S}_B$", r"$\mathcal{S}_C$"]          # the label of each panel
grid = np.linspace(-4, 4, 200)             # the pulls at which the standard normal is drawn
PAPER_STYLE = {"font.size": 10.0, "xtick.labelsize": 9.0, "ytick.labelsize": 9.0, "axes.linewidth": 0.5,
               "xtick.direction": "in", "ytick.direction": "in"}
with plt.rc_context(PAPER_STYLE):                                         # the paper's style, this plot only
    fig, axes = plt.subplots(1, 6, figsize=(12.0, 2.4), sharex=True, sharey=True)
    for k, ax in enumerate(axes):          # k: the panel's number
        # the panels 0 to 3: B's four parameters; the panels 4 and 5: the two flux ratios
        if k < 4:
            panel_pulls = pulls[:, k]
        else:
            panel_pulls = ratio_pulls[:, k - 4]
        ax.hist(panel_pulls, bins=np.linspace(-4, 4, 25), density=True, color="0.7")
        # the standard normal density
        ax.plot(grid, np.exp(-0.5 * grid ** 2) / np.sqrt(2 * np.pi), "k-", lw=0.8)
        ax.text(0.05, 0.95, names[k], transform=ax.transAxes, va="top", fontsize=9)
        ax.set_xlabel("pull")
    fig.tight_layout()
    plt.show()